In [0]:
def write_delta(
    df: DataFrame,
    base_path: str,
    table_name: str,
    merge_keys: list[str] | None = None,
    write_mode: str = "merge"
) -> None:
    """
    Escreve um DataFrame em formato Delta.

    Parameters
    ----------
    df : DataFrame
        DataFrame que será gravado.

    base_path : str
        Caminho base da camada (BRONZE, SILVER ou GOLD).

    table_name : str
        Nome da tabela.

    merge_keys : list[str], optional
        Chaves utilizadas para o MERGE.

    write_mode : str, optional
        "merge" (default) ou "overwrite".
    """

    path = f"{base_path}/{table_name}"

    # =====================================================
    # OVERWRITE
    # =====================================================
    if write_mode.lower() == "overwrite":

        (
            df.write
            .format("delta")
            .mode("overwrite")
            .option("overwriteSchema", "true")
            .save(path)
        )

        return

    # =====================================================
    # MERGE
    # =====================================================

    if merge_keys is None or len(merge_keys) == 0:
        raise ValueError(
            "merge_keys deve ser informado quando write_mode='merge'."
        )

    # Se a tabela ainda não existir, cria
    if not DeltaTable.isDeltaTable(spark, path):

        (
            df.write
            .format("delta")
            .mode("overwrite")
            .option("overwriteSchema", "true")
            .save(path)
        )

        return

    delta_table = DeltaTable.forPath(spark, path)

    merge_condition = " AND ".join(
        [
            f"target.{key} = source.{key}"
            for key in merge_keys
        ]
    )

    (
        delta_table.alias("target")
        .merge(
            df.alias("source"),
            merge_condition
        )
        .whenMatchedUpdateAll()
        .whenNotMatchedInsertAll()
        .execute()
    )